# ES-FR Border: Flow Position & Distribution

**Before you run this:** sign up at [voltpath.ai/dashboard](https://voltpath.ai/dashboard) for a free personal API key (copy it any time from [voltpath.ai/settings](https://voltpath.ai/settings)), then copy `.env.example` to `.env` and paste it in as `VOLTPATH_API_KEY`.

**Data source:** [ENTSO-E Transparency Platform](https://transparency.entsoe.eu), via the [Voltpath API](https://voltpath.ai) (docs at [docs.voltpath.ai](https://docs.voltpath.ai)).

In [ ]:
# common.py is shared across every article and changes often; without this a
# kernel that already imported it keeps serving the stale module.
%load_ext autoreload
%autoreload 2

import sys
from datetime import UTC, datetime, timedelta
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

from common import fetch_flows_range, flow_duration_curve, flow_type_shares, flow_type_stacked_bar, save_chart

In [ ]:
end = datetime.now(UTC)
start = end - timedelta(days=365)
date_from, date_to = start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d")

flows = fetch_flows_range(date_from, date_to, zone=["ES", "FR", "PT"])
flows = flows[flows["border"].isin(["ES -> FR", "FR -> ES", "ES -> PT", "PT -> ES"])]
flows["border"].unique()

## Flow duration curve

In [ ]:
fig = flow_duration_curve(flows, f"ES-FR / ES-PT flow duration, {date_from} to {date_to}")
save_chart(fig, f"flow-duration-ES-FR-ES-PT_{date_from}_{date_to}")
fig.show()

## Flow types per border

In [ ]:
shares = flow_type_shares(flows)
fig = flow_type_stacked_bar(shares, f"Flow types per border, {date_from} to {date_to}")
save_chart(fig, f"flow-types-ES-FR-ES-PT_{date_from}_{date_to}")
fig.show()

## Net position by month

In [ ]:
from common import flow_type_shares, net_position_bar, net_position_by_period

monthly_position = net_position_by_period(flows, "M")
month_order = sorted(monthly_position["period"].unique())
fig = net_position_bar(
    monthly_position, f"ES-FR / ES-PT net position by month, {date_from} to {date_to}", month_order
)
save_chart(fig, f"net-position-by-month-ES-FR-ES-PT_{date_from}_{date_to}")
fig.show()

## Flow types by month

In [ ]:
from common import flow_type_stacked_bar_by_period

fig = flow_type_stacked_bar_by_period(flows, "M", f"Flow types by month, {date_from} to {date_to}")
save_chart(fig, f"flow-types-by-month-ES-FR-ES-PT_{date_from}_{date_to}")
fig.show()

## Net position by season

In [ ]:
from common import SEASON_ORDER

seasonal_position = net_position_by_period(flows, "season")
fig = net_position_bar(
    seasonal_position, f"ES-FR / ES-PT net position by season, {date_from} to {date_to}", SEASON_ORDER
)
save_chart(fig, f"net-position-by-season-ES-FR-ES-PT_{date_from}_{date_to}")
fig.show()

## Flow types by season

In [ ]:
fig = flow_type_stacked_bar_by_period(flows, "season", f"Flow types by season, {date_from} to {date_to}")
save_chart(fig, f"flow-types-by-season-ES-FR-ES-PT_{date_from}_{date_to}")
fig.show()